# Repack NanoAOD ROOT files over Dask + XRootD

Distribute ROOT file repacking across Dask workers. Each worker:

1. Streams source files straight from XRootD (no local input copy).
2. Slices / re-baskets / re-compresses via the vendored `root_repack`. Sliced or rewritten segments hit local scratch; whole-file pass-throughs are fed to `TFileMerger` by URL.
3. Writes the merged output to local scratch (XRootD doesn't support the random-access writes `TFileMerger` needs to finalise a file), then `xrdcp`s to `<dst>.tmp` and `xrdfs mv`s into place.
4. Cleans up local scratch on task exit (bounded by `MAX_SCRATCH_GB`).

Driver side loads the input fileset JSON (with pre-computed `nevts` per file), plans one output file per chunk, ships the `intccms` package to scheduler/workers via `client.upload_file`, runs tasks with a tqdm progress bar, and writes an equivalent output-fileset JSON at the end.

## Analysis facility

In [ ]:
AF = "coffeacasa-condor"  # [coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
AUTO_CLOSE_CLIENT = False

## Configuration

Every knob the CLI exposes is here. `BASKET_SIZE` / `BASKET_SIZES` / `AUTO_FLUSH` trigger a rewrite pass (more scratch usage); leave as `None` for a plain merge/split.

In [ ]:
# --- Inputs ---
INPUT_JSON = "~/intc/integration-challenge/cms/example_cms/outputs/metadata/nanoaods.json"

# --- Outputs ---
OUTPUT_DIR_URL = "root://xrootd-local.unl.edu:1094//store/user/IC/repack_out"
OUTPUT_SUBDIR = "{dataset}/{systematic}"  # under OUTPUT_DIR_URL; files become 001.root, 002.root, ...
OUTPUT_FILESET_JSON = "outputs/repack/nanoaods_repacked.json"

# --- Split / merge ---
N_EVENTS = 1_000_000      # None = one output per (dataset, systematic); int = cap per output
EVENT_TREE = "Events"

# --- Tree re-encoding (triggers a rewrite pass per input, increases scratch usage) ---
BASKET_SIZE = None         # single size for all branches, e.g. "64k"
BASKET_SIZES = None        # list of patterns, e.g. ["Muon_*=128k", "Jet_*=256k"]
AUTO_FLUSH = None          # e.g. "30M" (byte threshold) or an int (entry count)

# --- TFileMerger options ---
FAST = False               # ROOT fast-merge mode
KEEP = False               # keep input compression instead of re-compressing
SORT = "branch"            # branch | offset | entry
COMPRESS = "same"          # e.g. "zstd=9", "lz4", "same"
IOFEATURES = None          # e.g. ["GenerateOffsetMap"]
VERBOSE = 0

# --- Worker scratch ---
SCRATCH_ROOT = "/tmp/intccms_repack"
MAX_SCRATCH_GB = 7.0       # peak per task ~ 2 x chunk_size for sliced chunks; stay below worker quota

# --- Runtime ---
OVERWRITE = False          # overwrite existing outputs on XRootD (xrdcp -f + mv)
PROGRESS = True
RAISE_ON_ERROR = True      # if False, failures are returned as exceptions in the results dict

# --- Dry run ---
DRY_RUN = False            # True = plan + summarize only; skip mkdir, dask submission, output JSON

## Imports and cloudpickle registration

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

In [ ]:
import cloudpickle

import intccms
from intccms.utils.dask_client import acquire_client
from intccms.utils.repack import (
    load_fileset,
    plan_chunks,
    prepare_output_dirs,
    run_repack,
    write_output_fileset,
)

cloudpickle.register_pickle_by_value(intccms)

In [ ]:
import shutil

# Built once on the driver. Shipped to the scheduler + every worker via
# client.upload_file() inside the run cell, which puts intccms on their
# sys.path so deserialization of the task graph works.
INTCCMS_ZIP = shutil.make_archive("/tmp/intccms_pkg", "zip", root_dir=str(src_dir), base_dir="intccms")
print(f"built {INTCCMS_ZIP}")

## Load fileset and plan chunks

In [ ]:
files = load_fileset(INPUT_JSON)
ds_syst_pairs = {(f.dataset, f.systematic) for f in files}
print(f"{len(files)} input files across {len(ds_syst_pairs)} dataset/systematic pairs")
print(f"total events: {sum(f.nevts for f in files):,}")

In [ ]:
plans = plan_chunks(
    files,
    output_dir_url=OUTPUT_DIR_URL,
    n_events=N_EVENTS,
    output_subdir=OUTPUT_SUBDIR,
)
print(f"{len(plans)} output chunks planned, {sum(p.total_events for p in plans):,} total events")
for plan in plans[:5]:
    print(f"  {plan.output_url}  ({plan.total_events:,} events from {len(plan.unique_sources)} source files)")
if len(plans) > 5:
    print(f"  ... ({len(plans) - 5} more)")

In [ ]:
from collections import Counter

per_ds = Counter((p.dataset, p.systematic) for p in plans)
event_counts = [p.total_events for p in plans]
segment_counts = [len(p.segments) for p in plans]
source_counts = [len(p.unique_sources) for p in plans]

print(f"chunks: {len(plans)}")
print(f"events per chunk:   min={min(event_counts):,}  max={max(event_counts):,}  mean={sum(event_counts) // len(event_counts):,}")
print(f"segments per chunk: min={min(segment_counts)}  max={max(segment_counts)}")
print(f"sources  per chunk: min={min(source_counts)}  max={max(source_counts)}   (= max xrdcp ops per task)")

top = sorted(per_ds.items(), key=lambda kv: -kv[1])[:10]
print("\ntop-10 (dataset, systematic) by chunk count:")
for (ds, syst), n in top:
    print(f"  {ds}/{syst}: {n} chunks")

In [ ]:
if DRY_RUN:
    n_dirs = len({p.output_url.rsplit("/", 1)[0] for p in plans})
    print(f"[dry run] would mkdir -p {n_dirs} output directories on XRootD")
else:
    prepare_output_dirs(plans)
    print(f"mkdir -p done for {len({p.output_url.rsplit('/', 1)[0] for p in plans})} output directories")

## Run the repack

In [ ]:
if DRY_RUN:
    print(f"[dry run] would submit {len(plans)} tasks to {AF}; skipping")
    results = {}
else:
    with acquire_client(AF, close_after=AUTO_CLOSE_CLIENT) as (client, cluster):
        client.upload_file(INTCCMS_ZIP)
        results = run_repack(
            client,
            plans,
            scratch_root=SCRATCH_ROOT,
            max_scratch_gb=MAX_SCRATCH_GB,
            overwrite=OVERWRITE,
            event_tree=EVENT_TREE,
            basket_size=BASKET_SIZE,
            basket_sizes=BASKET_SIZES,
            auto_flush=AUTO_FLUSH,
            fast=FAST,
            keep=KEEP,
            sort=SORT,
            compress=COMPRESS,
            iofeatures=IOFEATURES,
            verbose=VERBOSE,
            progress=PROGRESS,
            raise_on_error=RAISE_ON_ERROR,
        )

    n_ok = sum(1 for v in results.values() if isinstance(v, str))
    n_fail = len(results) - n_ok
    print(f"wrote {n_ok}/{len(results)} outputs ({n_fail} failed)")
    for url, outcome in results.items():
        if not isinstance(outcome, str):
            print(f"  FAIL {url}: {outcome!r}")

## Write output fileset JSON

Same shape as the input JSON, listing only chunks that succeeded. Feed this back into downstream tools (coffea preprocessing, the analysis notebooks, etc.).

In [ ]:
if DRY_RUN:
    print(f"[dry run] would write output fileset JSON to {OUTPUT_FILESET_JSON}")
else:
    out_json = write_output_fileset(plans, results, OUTPUT_FILESET_JSON)
    print(f"output fileset JSON: {out_json}")